# Spring 2026 Machine Vision (Graduate) Final Course Report

**Name**: ________________

**Student ID**: ________________

**Class**: ________________


---

## Assessment Instructions

1. **Assessment Format**: Prompt-based AI-assisted programming + Jupyter Notebook embedded auto-grading.
2. **Submission Method**: After running all cells, export this `.ipynb` file to **HTML** then convert to **PDF** document, and upload both the **ipynb** and **PDF** formats to the exam platform.
3. **Academic Integrity**:
   - You are allowed to use any large language model tools to assist in generating code;
   - You **must** paste the Prompts you used in the designated areas;
   - Manual debugging and modification of AI-generated code is permitted; you are encouraged to document your modification rationale in the report.
4. **Grading Composition**: Objective performance score (80 points, system-calculated) + Subjective analysis score (20 points, instructor-graded).

> **Note**: This Notebook will generate a personalized random seed based on your **Student ID**. Directly copying code from others may cause it to fail in your environment.

## 1. Environment Initialization

In [ ]:
# =================================================================
# Prepare the dataset and runtime environment; comment out after setup
# =================================================================
# !unzip ./data/Images.zip -d ./data
# !unzip ./data/Annotations.zip -d ./data
# !pip install tqdm
# !pip install seaborn



### 1.1 Student ID Input

Please enter your student ID in the code cell below to bind the random seed and test set to your identity.

In [ ]:
# ========== Enter your Student ID here ==============================
STUDENT_ID = "2023123456"  # Example student ID; replace with your own
# ====================================================================

### 1.2 Import Dependencies

Import all Python packages required for this experiment.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Import experiment dependencies
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score
import json

print("[INFO] All dependencies imported successfully")

### 1.3 Random Seed and GPU Configuration

Set the global random seed based on your student ID to ensure reproducibility; also detect GPU availability.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Set global random seed based on student ID
# and check GPU availability
# ============================================================

def set_random_seed(student_id: str):
    """Generate a global random seed from the student ID string."""
    digits = ''.join(filter(str.isdigit, str(student_id)))
    if not digits:
        digits = '123456'
    seed = int(digits) % (2**31)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    return seed

SEED = set_random_seed(STUDENT_ID)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Student ID bound: {STUDENT_ID}, Random seed: {SEED}")
print(f"[INFO] Using device: {device}")

## 2. Dataset Construction

### 2.1 MIT Indoor67 Dataset Introduction

MIT Indoor67 is a widely-used image dataset for indoor scene classification and recognition, released by the MIT Computer Science and Artificial Intelligence Laboratory (MIT CSAIL). The dataset is designed to support indoor scene recognition tasks and is widely used in robotic vision, environmental perception, image retrieval, and deep learning training. The dataset contains 67 categories of indoor scenes with a total of 15,620 images. Each category contains approximately 100-150 images of varying sizes, all of which are real-world photographs, mostly sourced from the internet.

In this project, image samples are located in the `./data/Images/` directory, with one folder per category named after the category. The structure is as follows:

```plaintext
./
└── data/
    └── Images/
        ├── greenhouse/          # Greenhouse image samples
        ├── florist/             # Florist image samples
        ├── computerroom/        # Computer room image samples
        ├── cloister/            # Cloister image samples
        ├── movietheater/        # Movie theater image samples
        ├── auditorium/          # Auditorium image samples
        ├── kindergarden/        # Kindergarten image samples
        ├── concert_hall/        # Concert hall image samples
        ├── church_inside/       # Church interior image samples
        └── library/             # Library image samples
```

### 2.2 Custom Dataset Class

The `MITIndoorDataset` class below is **provided code** and does not need modification. It reads image paths from `train_simple.txt` and `test_simple.txt`, and retains only the specified 10 categories.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Load the MIT Indoor 67 dataset with preprocessing
# ============================================================

class MITIndoorDataset(Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        ALLOWED_CATEGORIES = [
            'greenhouse', 'florist', 'computerroom', 'cloister', 'movietheater',
            'auditorium', 'kindergarden', 'concert_hall', 'church_inside', 'library'
        ]
        with open(txt_file, 'r') as f:
            lines = [line.strip() for line in f.readlines()]
        self.image_paths = []
        for line in lines:
            full_path = os.path.join(root_dir, line)
            dir_path = os.path.dirname(os.path.normpath(full_path))
            class_name = os.path.basename(dir_path)
            if class_name in ALLOWED_CATEGORIES:
                self.image_paths.append(full_path)
        self.classes = ALLOWED_CATEGORIES
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.labels = []
        for path in self.image_paths:
            dir_path = os.path.dirname(os.path.normpath(path))
            class_name = os.path.basename(dir_path)
            self.labels.append(self.class_to_idx[class_name])
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

### 2.3 Prompt Task: Data Augmentation

#### Task Description

The MIT Indoor67 dataset presents the following challenges when used in practice:

1. **Limited training data with blurry class boundaries**: Only about 100-150 images per category, severely constraining deep learning model training; visual similarity exists between different categories (e.g., computerroom vs. office).
2. **Inconsistent image resolution and quality**: Images come from diverse sources with significant differences in resolution, clarity, and lighting conditions.
3. **Relatively uniform but unrealistic class distribution**: In real-world scenarios, certain room types are far more common than others.

**Requirements**:
- `train_transform` must include **at least 3** data augmentation methods (e.g., random cropping, horizontal flipping, color jittering, random rotation, random erasing, etc.), followed by normalization;
- `val_transform` should only include basic Resize + CenterCrop + Normalize without data augmentation;
- Use PyTorch's `torchvision.transforms` for implementation.

#### [Example Prompt]

```text
Please use the PyTorch deep learning framework to write an image data preprocessing pipeline (data transforms) for image classification. Abbreviate torchvision.transforms as T. Please generate a dictionary named data_transforms with 'train' and 'val' keys.
For the 'train' training set, include the following data augmentation operations in order:
- Randomly crop and resize to 224 (FUNNAME)
- Random horizontal flip (FUNNAME)
- Color jitter: brightness, contrast, saturation all 0.2, hue 0.1 (FUNNAME)
- Random rotation by 15 degrees (FUNNAME)
- Random vertical flip (FUNNAME)
- Convert to Tensor (FUNNAME)
- Random erasing (FUNNAME)
- Normalize using ImageNet mean ([0.485, 0.456, 0.406]) and standard deviation ([0.229, 0.224, 0.225]) (FUNNAME)
For the 'val' validation set, include the following operations in order:
- Resize image to 256 (FUNNAME)
- Center crop to 224 (FUNNAME)
- Convert to Tensor (FUNNAME)
- Normalize using the same ImageNet parameters as the training set (FUNNAME)
Please output only the code.
```

> **Student Fill-in Area**

(Please paste the Prompt you actually used here)

In [ ]:
# ============================================================
# Data augmentation code implementation area (teacher reference)
# ============================================================

# ================== [Student Code Paste Area] ==================
# Paste your data augmentation code here.
# The dictionary name must remain data_transforms for subsequent calls.

data_transforms = {
    'train': None,  # Placeholder; replace with your actual training transformations
    'val': None     # Placeholder; replace with your actual validation transformations
}

### 2.4 Split Training and Validation Sets

Construct the training and validation sets using `train_simple.txt` and `test_simple.txt`. Note: In practice, the dataset should typically be split into training, validation, and test sets. Here, for simplicity, the validation and test sets are combined.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Split the dataset
# ============================================================

data_dir = './data'
image_datasets = {
    'train': MITIndoorDataset(
        txt_file=os.path.join(data_dir, 'train_simple.txt'),
        root_dir=os.path.join(data_dir, 'Images'),
        transform=data_transforms['train']
    ),
    'val': MITIndoorDataset(
        txt_file=os.path.join(data_dir, 'test_simple.txt'),
        root_dir=os.path.join(data_dir, 'Images'),
        transform=data_transforms['val']
    )
}

class_names = image_datasets['train'].classes
num_classes = len(class_names)
print(f"[INFO] Training set size: {len(image_datasets['train'])}, Validation set size: {len(image_datasets['val'])}")
print(f"[INFO] Class list: {class_names}")

## 3. Model Design

### 3.1 Prompt Task: Model Architecture

#### Task Description

Design a deep learning model for indoor scene classification. Requirements:
- Must use a **pretrained backbone** (e.g., ResNet-18, VGG, etc.) as the feature extractor;
- Build a **custom classification head** (with at least one hidden fully-connected layer);
- Apply proper **parameter initialization** to the custom layers (e.g., Kaiming Normal);
- Implement using the PyTorch framework, inheriting from `torch.nn.Module`.

#### [Example Prompt]

```text
Please use the PyTorch framework to build a deep learning network model for indoor scene classification with the following specifications:
1. Model class name: IndoorSceneClassifier
2. Network architecture:
   - Feature extractor: XXXXNet backbone, outputting a 512-dimensional feature vector
   - Classification head: Three cascaded fully-connected layers
     * FC1: 512 → 1024, ReLU activation
     * FC2: 1024 → 512, ReLU activation
     * FC3: 512 → 256, ReLU activation
   - Output layer: 256 → 10 classes
3. Parameter initialization: All fully-connected layers should use He Kaiming initialization (FUNNAME)
4. Code conventions: Inherit from FUNAME, include complete FUNNAME and forward methods, add comments.
Please provide a complete, runnable Python code implementation.
```

> **Student Fill-in Area**

(Please paste the Prompt you actually used here)

In [ ]:
# ============================================================
# Model architecture code implementation area
# ============================================================

# ================== [Student Code Paste Area] ==================
# Paste your AI-generated model building code here.
# The class name must remain IndoorSceneClassifier for subsequent training and grading.

class IndoorSceneClassifier(nn.Module):
    """
    Indoor scene classification model based on ResNet-18 feature extractor
    and multi-layer fully-connected classification head.
    Architecture: ResNet-18 (backbone) → FC1 (512→1024) → FC2 (1024→512) → FC3 (512→256) → Output (256→10)
    """
    def __init__(self, num_classes=10):
        super(IndoorSceneClassifier, self).__init__()
        pass  # Placeholder; replace with your actual model initialization code

    def forward(self, x):
        pass    # Placeholder; replace with your actual forward pass code
        return out

    def print_params(self):
        pass  # Placeholder; replace with your actual parameter printing code



In [ ]:
# ============================================================
# [DO NOT MODIFY] Instantiate the model
# ============================================================

# Instantiate the model and move to device
model = IndoorSceneClassifier(num_classes=num_classes)
model = model.to(device)

### 3.2 Model Architecture Diagram

**TODO: Please paste your model architecture diagram below.**

You can use Draw.io, Visio, PowerPoint, or other drawing tools to create the diagram, export it as an image, and reference it here:

```markdown
<div style="text-align:center">
  <img src="model_structure.png" alt="Model Architecture Diagram" width="600px">
  <p>Figure 3-1 Network model architecture diagram</p>
</div>
```

> Note: Please name your model architecture diagram `model_structure.png` and save it in the current directory.

## 4. Model Training

### 4.1 Loss Function and Optimizer

The model uses **Cross Entropy Loss** as the loss function, defined as:

$$
\mathcal{L}_{\text{CE}} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{i,c} \log(p_{i,c})
$$

where $N$ is the number of samples, $C$ is the number of classes, $y_{i,c}$ is the true label of sample $i$ (one-hot encoded), and $p_{i,c}$ is the model's predicted probability that sample $i$ belongs to class $c$.

The optimizer chosen is **Adam**, with an initial learning rate typically set around $10^{-3}$. Adam combines the advantages of momentum and RMSProp, adaptively adjusting the learning rate for each parameter to accelerate convergence.

### 4.2 Hyperparameter Tuning

There are three key hyperparameters that can be tuned during model training: **learning rate**, **batch size**, and **number of epochs**.

- **Learning rate**: Controls the step size of parameter updates. Too large may cause oscillation or divergence; too small leads to slow convergence.
- **Batch size**: Affects the noise level of gradient estimation and memory usage. Larger batches produce more stable gradients but require more GPU memory.
- **Number of epochs**: The number of times the model traverses the complete training data. Too few leads to underfitting; too many may cause overfitting.

In [ ]:
# ============================================================
# Hyperparameter settings — adjust here
# ============================================================

batch_size = 16        # Batch size: balancing gradient stability and GPU memory
learning_rate = 0.001  # Initial learning rate: common default for Adam
num_epochs = 5         # Number of epochs: ensure sufficient convergence

print(f"[INFO] Hyperparameter settings: batch_size={batch_size}, lr={learning_rate}, epochs={num_epochs}")

### 4.3 Create DataLoader

In [ ]:
# ============================================================
# [DO NOT MODIFY] Create DataLoader instances
# ============================================================

dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True, num_workers=0),
    'val': DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False, num_workers=0)
}
print("[INFO] DataLoaders created successfully")

### 4.4 Prompt Task: Define Loss Function and Optimizer

#### Task Description

Configure the loss function and optimizer required for deep learning model training. Requirements:

- Use cross-entropy loss as the evaluation criterion for the classification task;

- Configure the Adam optimizer, correctly passing the model parameters and learning rate;

- Include informative console print statements for debugging and progress tracking;

- Implement using the PyTorch framework.

#### [Example Prompt]

```text
Please use PyTorch to configure the loss function and optimizer for deep learning model training with the following specifications:
1. Loss function: Instantiate cross-entropy loss (FUNNAME) and assign it to the variable criterion.
2. Optimizer: Instantiate the Adam optimizer (FUNNAME) and assign it to the variable optimizer, with the following parameters:
    - Model parameters: model.parameters()
    - Learning rate: learning_rate
3. Logging: After execution, print the message: "[INFO] Loss function and optimizer initialized successfully".
4. Code conventions: Provide a complete, concise Python code snippet. No extra model definition needed; assume model and learning_rate are already defined in context.
```

> **Student Fill-in Area**

(Please paste the Prompt you actually used here)

In [ ]:
# ============================================================
# Configure the loss function and optimizer for model training
# ===========================================================

# ================== [Student Code Paste Area] ==================
# Paste your AI-generated code here.
# The loss function variable must be named criterion, and the optimizer variable must be named optimizer for subsequent training and grading.

criterion = None  # Placeholder; replace with your actual loss function instantiation code
optimizer = None  # Placeholder; replace with your actual optimizer instantiation code
print("[INFO] Loss function and optimizer initialized successfully")



### 4.5 Training Function

The training function below is **provided code** and does not need modification. During training, the model goes through training (train) and validation (val) phases in each epoch. In the training phase, the model performs forward propagation, computes loss, backpropagation, and parameter optimization for each batch. The validation phase only performs forward propagation and computes performance metrics without gradient updates. To save the best-performing model parameters, the function checks whether the current validation accuracy exceeds the historical best accuracy during the validation phase, and if so, saves the current model parameters.

In [ ]:
# ============================================================
# [Provided Code] Main training loop — do not modify unless necessary
# ============================================================

def train_model(model, dataloaders, criterion, optimizer, num_epochs=25):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs-1}')
        print('-' * 10)
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()
            running_loss = 0.0
            running_corrects = 0
            if phase == 'train':
                data_iter = tqdm(dataloaders[phase], total=len(dataloaders[phase]),
                                desc=f'Epoch {epoch} {phase}', unit='batch')
            else:
                data_iter = dataloaders[phase]
            for batch_id, data in enumerate(data_iter):
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    preds = torch.argmax(outputs, 1)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data).item()
            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects / len(dataloaders[phase].dataset)
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc)
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), 'best_model.pth')
    print(f'Best val Acc: {best_acc:4f}')
    return model, history

### 4.6 Training Visualization Function

In [ ]:
# ============================================================
# [Provided Code] Do not modify unless necessary
# ============================================================

def plot_training_history(history):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

### 4.7 Run Training

In [ ]:
# ============================================================
# [DO NOT MODIFY] Run model training
# ============================================================

model, history = train_model(model, dataloaders, criterion, optimizer, num_epochs=num_epochs)
plot_training_history(history)

## 5. Model Evaluation

### 5.1 Evaluation Function

The evaluation function below is **provided code**. It computes the confusion matrix, classification report (Precision, Recall, F1), per-class average precision (AP), and overall mAP.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Model evaluation code implementation
# ============================================================

def evaluate_model(model, dataloader, class_names):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch_id, data in enumerate(dataloader):
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            probs = F.softmax(outputs, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
    cm = confusion_matrix(all_labels, all_preds)
    cr = classification_report(all_labels, all_preds, target_names=class_names)
    y_bin = np.zeros((len(all_labels), len(class_names)))
    y_bin[np.arange(len(all_labels)), all_labels] = 1
    ap_per_class = average_precision_score(y_bin, np.array(all_probs), average=None)
    map_score = np.mean(ap_per_class)
    class_ap = [(class_names[i], ap) for i, ap in enumerate(ap_per_class)]
    class_ap_sorted = sorted(class_ap, key=lambda x: x[1], reverse=True)
    ap_report = "Per-class AP (sorted descending):\n"
    for class_name, ap in class_ap_sorted:
        ap_report += f"{class_name}: {ap:.4f}\n"
    return cm, cr, map_score, ap_report

### 5.2 Confusion Matrix

The confusion matrix constructs a 2D matrix with true labels as rows and predicted labels as columns. Diagonal values represent correctly classified samples, while off-diagonal values represent misclassified samples. The confusion matrix provides an intuitive view of which classes the model performs well on and which classes it tends to confuse.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Confusion matrix visualization function
# ============================================================

def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Load the best model and evaluate
model.load_state_dict(torch.load('best_model.pth', map_location=device))
cm, cr, map_score, ap_report = evaluate_model(model, dataloaders['val'], class_names)
plot_confusion_matrix(cm, class_names)

### 5.3 Classification Report and Evaluation Metrics

Common metrics for evaluating classification models include **Precision**, **Recall**, **F1-score**, and **mAP (mean Average Precision)**.

**Precision** measures the proportion of predicted positive samples that are actually positive:

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Recall** measures the proportion of actual positive samples that are correctly predicted:

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**F1-score** is the harmonic mean of precision and recall:

$$
\text{F1-score} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

**mAP (mean Average Precision)** is a comprehensive metric for multi-class classification. First, AP (area under the PR curve) is computed for each class, then averaged:

$$
\text{mAP} = \frac{1}{N} \sum_{i=1}^N \text{AP}_i
$$

where $N$ is the number of classes and $\text{AP}_i$ is the average precision for class $i$.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Print the classification report
# ============================================================

print("\nClassification Report:")
print(cr)
print(f"\nmAP Score: {map_score:.4f}")
print("\n" + ap_report)

## 6. Model Deployment and Inference

### 6.1 Model Loading Instructions

The trained model parameters are saved in a `.pth` file. For deployment, you first need to create a model instance using the model class definition to construct the model architecture, then load the parameter file, and finally set the model to evaluation mode (`model.eval()`) before it can be used for inference.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Load test samples
# ============================================================

def load_test_samples(txt_path):
    samples = []
    with open(txt_path, 'r') as f:
        for line in f:
            if ' ' in line:
                parts = line.strip().split()
                if len(parts) >= 2:
                    img_path = parts[0]
                    label = int(parts[1])
                    samples.append((img_path, label))
            else:
                folder = line.strip().split('/')[0]
                label_mapping = {
                    'greenhouse': 0, 'florist': 1, 'computerroom': 2,
                    'cloister': 3, 'movietheater': 4, 'auditorium': 5,
                    'kindergarden': 6, 'concert_hall': 7,
                    'church_inside': 8, 'library': 9
                }
                label = label_mapping.get(folder, -1)
                if label != -1:
                    samples.append((line.strip(), label))
    return samples

# Validation set preprocessing (for inference)
val_transform_deploy = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load test samples
test_samples = load_test_samples('./data/test_simple.txt')
print(f"[INFO] Number of test samples loaded: {len(test_samples)}")

### 6.2 Prompt Task: Model Deployment and Inference

#### Task Description

Write a function `recognize_location(model, data_dir, test_samples)` that implements the following:
- Randomly select an image from the test sample list;
- Preprocess the image (using the validation transform);
- Perform forward inference with the model to obtain the predicted class;
- Return the original image, predicted label, true label, and image path.

Use OpenCV to read the image and convert it to RGB format.

#### [Example Prompt]

```text
Please write an indoor scene inference function using PyTorch: recognize_location(model, data_dir, test_samples):
- Randomly select a sample from the test_samples list (format: (img_path, label));
- Use FUNNAME to read the image and convert to RGB;
- Convert the numpy array to a PIL Image, then apply val_transform for preprocessing (add batch dimension);
- Set the model to eval mode, use FUNNAME for inference;
- Return the original image (numpy array), predicted label (int), true label (int), image path (str).
- If image reading fails, return (None, None, None, None) and print a warning.
```

> **Student Fill-in Area**

(Please paste the Prompt you actually used here)

In [ ]:
# ============================================================
# Model deployment and inference code implementation
# ============================================================

# ================== [Student Code Paste Area] ==================
# Paste your AI-generated code here.
# The function must remain named recognize_location(model, data_dir, test_samples) for subsequent training and grading.

def recognize_location(model, data_dir, test_samples):
    pass  # Placeholder; replace with your actual inference code
    return img, int(pred_label), int(true_label), img_path

### 6.3 Run Inference Test

Randomly select 10 images from the test set for inference and count the number of correct predictions.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Model inference process
# ============================================================

label_names = {
    0: "greenhouse", 1: "florist", 2: "computerroom",
    3: "cloister", 4: "movietheater", 5: "auditorium",
    6: "kindergarden", 7: "concert_hall",
    8: "church_inside", 9: "library"
}

correct_predicts = 0
for room_index in range(10):
    img, pred_label, true_label, img_path = recognize_location(model, './data/Images/', test_samples)
    print(f"Starting room {room_index} recognition...")
    if img is not None:
        plt.figure(figsize=(5, 4))
        plt.imshow(img)
        plt.title(f"room {room_index}\npredict: {label_names[pred_label]}, true: {label_names[true_label]}")
        plt.axis('off')
        plt.show()
        if pred_label == true_label:
            correct_predicts += 1
    else:
        print(f"Room {room_index} recognition failed")

print(f"\n[INFO] Inference test completed: {correct_predicts}/10 correct predictions")

## 7. Reflection and Summary (Subjective Questions, 20 points)

Please answer the following questions. Your answers will be included in the subjective scoring.

### Question 1

In your experience collaborating with large language models for programming, what types of logic do you think LLMs are least capable of handling? How did you modify your prompts to address or mitigate this issue?

> **Student Answer Area**

(Please write your answer here)

### Question 2

If your model's validation accuracy stops improving while the training loss continues to decrease, what does this typically indicate? What specific measures would you take to address this?

> **Student Answer Area**

(Please write your answer here)

### Question 3

Through this experiment, what new insights have you gained about the relationship between model architecture and performance? For example, how do the pretrained backbone, classification head design, and parameter initialization each affect the final results?

> **Student Answer Area**

(Please write your answer here)

## 8. Auto-Grading System

The auto-grading system below is a **restricted area**. The system will score objectively from four dimensions: model architecture, training performance, mAP metric, and deployment inference (total 80 points), with an additional 20 subjective points graded by the instructor.

In [ ]:
# ============================================================
# [DO NOT MODIFY] Auto-grading system
# ============================================================

def auto_grade(student_id, model, dataloaders, class_names, test_samples,
               data_dir, map_score, correct_predicts, num_classes=10):
    """
    Multi-dimensional auto-grading system
    Scoring breakdown:
    - Model architecture score (20 points): Evaluates model structural soundness
    - Training performance score (30 points): Based on validation set mAP improvement
    - mAP metric score (20 points): Based on test set mAP
    - Deployment inference score (10 points): Based on deployed inference accuracy
    - Total: 80 points (objective), plus 20 subjective points graded by instructor
    """
    results = {
        "student_id": student_id,
        "scores": {},
        "details": {}
    }

    # --- 1. Model architecture score (20 points) ---
    arch_score = 0
    arch_details = []

    # Check for pretrained backbone (5 points)
    has_backbone = False
    for name, module in model.named_modules():
        if 'resnet' in name.lower() or 'vgg' in name.lower() or 'backbone' in name.lower():
            has_backbone = True
            break
    if has_backbone:
        arch_score += 5
        arch_details.append("✓ Uses pretrained backbone: +5 points")
    else:
        arch_details.append("✗ No pretrained backbone detected: +0 points")

    # Check for custom classification head (5 points)
    fc_count = 0
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) and 'fc' in name.lower():
            fc_count += 1
    if fc_count >= 2:
        arch_score += 5
        arch_details.append(f"✓ Custom classification head ({fc_count} FC layers detected): +5 points")
    elif fc_count >= 1:
        arch_score += 3
        arch_details.append(f"△ Simple classification head (only {fc_count} FC layer): +3 points")
    else:
        arch_details.append("✗ No custom classification head detected: +0 points")

    # Check if model parameter count is reasonable (5 points)
    total_params = sum(p.numel() for p in model.parameters())
    if 1e6 <= total_params <= 50e6:
        arch_score += 5
        arch_details.append(f"✓ Reasonable parameter count ({total_params/1e6:.1f}M): +5 points")
    elif total_params < 1e6:
        arch_score += 2
        arch_details.append(f"△ Parameter count too low ({total_params/1e6:.1f}M): +2 points")
    else:
        arch_score += 3
        arch_details.append(f"△ Parameter count too high ({total_params/1e6:.1f}M): +3 points")

    # Check for parameter initialization (5 points)
    init_found = False
    for name, param in model.named_parameters():
        if 'fc' in name.lower() or 'output' in name.lower():
            init_found = True
            break
    if init_found:
        arch_score += 5
        arch_details.append("✓ Uses parameter initialization: +5 points")
    else:
        arch_details.append("✗ No parameter initialization detected: +0 points")

    results["scores"]["model_architecture"] = arch_score
    results["details"]["model_architecture"] = arch_details

    # --- 2. Training performance score (30 points) ---
    train_score = 0
    if map_score >= 0.85:
        train_score = 30
    elif map_score >= 0.75:
        train_score = 26
    elif map_score >= 0.65:
        train_score = 22
    elif map_score >= 0.55:
        train_score = 18
    elif map_score >= 0.45:
        train_score = 14
    elif map_score >= 0.35:
        train_score = 10
    elif map_score >= 0.25:
        train_score = 6
    else:
        train_score = 3

    results["scores"]["training_performance"] = train_score
    results["details"]["training_performance"] = f"mAP={map_score:.4f}, score={train_score}/30"

    # --- 3. mAP metric score (20 points) ---
    map_score_val = 0
    if map_score >= 0.80:
        map_score_val = 20
    elif map_score >= 0.70:
        map_score_val = 18
    elif map_score >= 0.60:
        map_score_val = 15
    elif map_score >= 0.50:
        map_score_val = 12
    elif map_score >= 0.40:
        map_score_val = 8
    elif map_score >= 0.30:
        map_score_val = 5
    else:
        map_score_val = 2

    results["scores"]["map_metric"] = map_score_val
    results["details"]["map_metric"] = f"mAP={map_score:.4f}, score={map_score_val}/20"

    # --- 4. Deployment inference score (10 points) ---
    deploy_score = min(correct_predicts * 1, 10)

    results["scores"]["deployment"] = deploy_score
    results["details"]["deployment"] = f"{correct_predicts}/10 correct predictions, score={deploy_score}/10"

    # --- Total score ---
    total_objective = arch_score + train_score + map_score_val + deploy_score
    results["scores"]["total_objective"] = total_objective
    results["scores"]["subjective_pending"] = 20
    results["scores"]["total_max"] = total_objective + 20

    return results

# Run grading
grading_results = auto_grade(
    student_id=STUDENT_ID,
    model=model,
    dataloaders=dataloaders,
    class_names=class_names,
    test_samples=test_samples,
    data_dir='./data/Images/',
    map_score=map_score,
    correct_predicts=correct_predicts,
    num_classes=num_classes
)

# Print grading report
print("=" * 60)
print("       Machine Vision Final Report - Auto-Grading Report")
print("=" * 60)
print(f"Student ID: {grading_results['student_id']}")
print("-" * 60)

for category, score in grading_results['scores'].items():
    if category in grading_results['details']:
        detail = grading_results['details'][category]
        if isinstance(detail, list):
            print(f"  {category}: {score} points")
            for d in detail:
                print(f"    {d}")
        else:
            print(f"  {category}: {score} points | {detail}")
    else:
        print(f"  {category}: {score} points")

print("-" * 60)
print(f"  Objective total: {grading_results['scores']['total_objective']}/80 points")
print(f"  Subjective pending: {grading_results['scores']['subjective_pending']}/20 points")
print(f"  Total maximum: {grading_results['scores']['total_max']}/100 points")
print("=" * 60)
print("\nNote: Subjective questions (prompt quality, model architecture diagram, experiment reflection) are graded by the instructor.")

## 9. Pre-Submission Checklist

- [ ] `STUDENT_ID` has been changed to your own student ID.
- [ ] All Prompt areas have your actual prompts pasted.
- [ ] Code areas have your debugged, runnable code pasted.
- [ ] Auto-grading has been run and the score report is generated and fully displayed.
- [ ] The Jupyter Notebook has been exported to HTML and converted to PDF.

> **Good luck with your exam!**